# Lecture 03 — Parsing HTML

> *"You don't really need a regex for HTML. You really, really don't."*

You have HTML bytes. You want structured data. The bridge between them is **a parser**: a piece of software that turns the messy, often-malformed string of HTML into a tree you can navigate.

## What you'll be able to do after this lecture

- Choose between `html.parser`, `lxml`, and `html5lib`, and explain why.
- Use BeautifulSoup's `find`, `find_all`, and `select` methods.
- Read and write CSS selectors fluently.
- Use XPath when CSS isn't enough.
- Write extractors that survive minor template changes.

## Setup

```bash
pip install beautifulsoup4 lxml httpx
```


## 1. Three parsers, one library

`BeautifulSoup` is a wrapper. Underneath it can use any of three parsers:

| Parser       | Speed | Lenient with broken HTML | Notes                              |
|--------------|-------|--------------------------|------------------------------------|
| `html.parser`| Slow  | Medium                   | Built into Python. No deps.        |
| `lxml`       | Fast  | High                     | C library. Installed via pip.      |
| `html5hib`   | Slow  | Highest (browser-grade)  | Pure Python. Best fidelity.        |

**Default recommendation:** `lxml`. It's fast, lenient enough for almost any real-world page, and the default in this course.

In [ ]:
from bs4 import BeautifulSoup

# Same input, three parsers
html = "<p>hi <b>there<p>second"  # deliberately broken

for parser in ["html.parser", "lxml", "html5lib"]:
    try:
        soup = BeautifulSoup(html, parser)
        print(f"{parser:12s} -> {soup}")
    except Exception as e:
        print(f"{parser:12s} -> not installed ({e})")


Notice each parser closes the unclosed tags differently. For most data extraction, `lxml` is fine. Where it bites you, fall back to `html5lib`.

## 2. Our practice site: books.toscrape.com

We'll do all examples in this lecture against [books.toscrape.com](https://books.toscrape.com/), a free static site explicitly built for scraping practice. It will not change under you, it will not block you, and it has predictable structure. Bookmark it.

In [ ]:
import httpx
from bs4 import BeautifulSoup

URL = "https://books.toscrape.com/"

resp = httpx.get(URL, timeout=10.0)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "lxml")

print(soup.title.string)


## 3. `find` and `find_all`

The classic BeautifulSoup API. `find(...)` returns the first match (or `None`). `find_all(...)` returns a list (possibly empty).

You can match by:

- **Tag name** — `find("h1")`, `find_all("a")`.
- **Attribute** — `find("a", href="/about")`.
- **CSS class** — `find_all(class_="price_color")` (note the underscore — `class` is reserved).
- **Multiple criteria** — `find_all("article", class_="product_pod")`.


In [ ]:
# Find every product card on the homepage
products = soup.find_all("article", class_="product_pod")
print(f"Found {len(products)} products on the page.\n")

# Pick one and extract title + price
first = products[0]
title = first.h3.a["title"]
price = first.find(class_="price_color").get_text(strip=True)
print("first product:", title, "--", price)


## 4. CSS selectors with `select`

`find_all` is fine but verbose. **CSS selectors** are the same query language your browser's DevTools speak, which means you can prototype them in the browser console (`document.querySelectorAll(...)`) and paste them straight into Python.

`soup.select(...)` returns a list of matches. `soup.select_one(...)` returns the first or `None`.

A cheat sheet:

| Selector                 | Matches                                                |
|--------------------------|--------------------------------------------------------|
| `tag`                    | Every `<tag>`.                                         |
| `.cls`                   | Every element with `class="cls"`.                      |
| `#id`                    | The element with `id="id"`.                            |
| `tag.cls`                | A `<tag>` with that class.                             |
| `parent child`           | Any `<child>` somewhere inside `<parent>`.             |
| `parent > child`         | A `<child>` that is a *direct* child of `<parent>`.    |
| `tag[attr]`              | Has the attribute at all.                              |
| `tag[attr="value"]`      | Has that attribute equal to that value.                |
| `tag[attr^="prefix"]`    | Attribute starts with prefix.                          |
| `tag[attr$="suffix"]`    | Attribute ends with suffix.                            |
| `tag[attr*="contains"]`  | Attribute contains substring.                          |
| `:nth-child(n)`          | The nth child. 1-indexed.                              |
| `tag:not(.cls)`          | A tag that *isn't* in that class.                      |


In [ ]:
# Same query as before, written with CSS selectors:
for product in soup.select("article.product_pod"):
    title = product.select_one("h3 a")["title"]
    price = product.select_one(".price_color").get_text(strip=True)
    in_stock = product.select_one(".availability").get_text(strip=True)
    print(f"{title:60.60s} {price:>8s}  {in_stock}")


### Finding the right selector

The honest workflow:

1. Open the page in your browser.
2. Right-click the element you want → **Inspect**.
3. In DevTools, right-click the highlighted node → **Copy → Copy selector**.
4. Paste that into your Python.
5. **Then** simplify the selector to the minimal piece that uniquely identifies what you want.

Step 5 matters. Browser-generated selectors are often `body > div:nth-child(3) > main > section > div.row > article:nth-child(1)` — brittle to any layout change. Prefer semantic anchors (class names that read like data: `.price`, `.product-title`) over positional ones.

## 5. Working with attributes and text

Every BeautifulSoup tag is a dict-like object for attributes and has helpers for text content.

In [ ]:
link = soup.select_one("article.product_pod h3 a")

# Attributes by indexing or .get
print("href:    ", link["href"])
print("title:   ", link.get("title"))
print("missing: ", link.get("nonexistent"))   # safe — returns None

# Text content variants
print("text:           ", repr(link.text))                      # raw, may include whitespace
print("text(strip=T):  ", repr(link.get_text(strip=True)))      # cleaned
print("all child text: ", repr(link.get_text(separator=" ", strip=True)))


## 6. Walking the tree

Sometimes the field you want is *near* an anchor element rather than inside it. BeautifulSoup gives you tree-walking helpers.

In [ ]:
price_tag = soup.select_one(".price_color")

# Up
print("parent:    ", price_tag.parent.name)
print("ancestors:", [t.name for t in price_tag.parents][:3])

# Sideways
print("next sib:  ", price_tag.find_next_sibling())
print("prev sib:  ", price_tag.find_previous_sibling())

# Down
container = price_tag.find_parent("article")
print("all <p>:   ", [p.get_text(strip=True) for p in container.find_all("p")])


These helpers are how you handle "the price is in the next `<span>` after the label" patterns that some lazily-templated sites use.

## 7. XPath — when CSS isn't enough

CSS selectors can't easily say *"the `<td>` whose text contains 'Availability'"* — they have no text predicate. **XPath** can. It's a more powerful query language used in many enterprise tools (and Selenium / Playwright happily accept it).

BeautifulSoup itself doesn't do XPath, but `lxml` does. The two libraries cooperate happily.

In [ ]:
from lxml import html

tree = html.fromstring(resp.text)

# Find every link whose href starts with 'catalogue/category'
category_links = tree.xpath("//a[starts-with(@href, 'catalogue/category')]/@href")
print("first 5 categories:", category_links[:5])

# Find a price by sibling-of-text-match (what CSS can't)
# (this works on a product detail page; demoing the syntax)
xpath_example = "//th[text()='Availability']/following-sibling::td/text()"
print("xpath example string:", xpath_example)


**Rule of thumb:** start with CSS selectors. They cover 95% of cases. Drop into XPath only when you need text matching, sibling navigation by content, or another XPath-only feature.

## 8. Robust extractors — designing for change

Selectors break. The site you're scraping today will rename its CSS classes, restructure its DOM, and move things around. Your extractor needs to fail loudly when it does, not silently.

### The defensive pattern

```python
def extract_product(card) -> dict:
    title_el = card.select_one("h3 a")
    if title_el is None:
        raise ValueError("product card has no title — selector likely outdated")
    title = title_el["title"]

    price_el = card.select_one(".price_color")
    price = price_el.get_text(strip=True) if price_el else None

    return {"title": title, "price": price}
```

Two principles:

1. **Required fields raise**, optional fields default to `None`. If `title` is missing, the page is unrecognizable; abort. If a "subtitle" is missing on some products, that's fine.
2. **Anchor on stable selectors.** Class names that look like *data* (`.price`, `.product-title`, `.availability`) tend to outlive class names that look like *layout* (`.col-3`, `.float-left`).

### The "two-selector fallback" pattern

When a site has two possible templates (older pages mixed with newer pages), try both:

```python
title = (
    card.select_one("h3.product-title a")
    or card.select_one("h3 a")
).get_text(strip=True)
```

Use sparingly — every fallback is selector debt that you'll have to maintain.


In [ ]:
# Putting it together: extract every product from the homepage as a list of dicts
def extract_product(card):
    a = card.select_one("h3 a")
    return {
        "title": a["title"] if a else None,
        "href": a["href"] if a else None,
        "price": (card.select_one(".price_color") or {}).get_text(strip=True) if card.select_one(".price_color") else None,
        "availability": (card.select_one(".availability") or {}).get_text(strip=True) if card.select_one(".availability") else None,
    }

records = [extract_product(c) for c in soup.select("article.product_pod")]
for r in records[:3]:
    print(r)
print(f"\ntotal: {len(records)}")


## 9. Encoding

Most modern sites are UTF-8 and `httpx`/BeautifulSoup figure it out automatically. When they don't, you'll see Korean/Chinese/Japanese text rendered as `ë§ˆì§€ë§‰` or `?????`.

Two debugging steps when this happens:
- `response.encoding` — what `httpx` thinks the encoding is.
- `response.headers.get("content-type")` — the server's claim.

If the server lies, force the right encoding: `response.encoding = "utf-8"`.

## Recap

- Default to `lxml` as your parser, fall back to `html5lib` for pathological HTML.
- `select`/`select_one` with CSS selectors is the modern way; `find`/`find_all` still works.
- Browser DevTools → Inspect → Copy selector is your fastest workflow, but **simplify** what it gives you.
- Required fields should raise loudly when missing. Optional fields default to `None`.
- Reach for XPath only when CSS can't express the query.

## Exercises

1. From `https://books.toscrape.com/`, extract every category in the left sidebar — return a list of `(name, url)` tuples.
2. From the same page, extract every product card as a list of dicts with `title`, `price`, `availability`, `image_url`. The image URL is on the `<img>` tag's `src` attribute.
3. Write a function `find_field(soup, label)` that takes a `<th>label</th><td>value</td>` table (common in product detail pages) and returns the value text for a given label. Test it on a single book detail page (e.g., `https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html`).
4. Visit `https://news.ycombinator.com/`. Extract the top 30 story titles and their points using only CSS selectors.

## Up next

**Lecture 04** brings everything from 02 and 03 together to build a complete crawler — multi-page, deduped, persistent, resumable. By the end you'll have a working tool, not just techniques.
